# 07 — Evaluate the frozen test for one release decision

**Plain-language question:** Does the exact chosen model work on later data it
did not use?

**Why this matters:** the test set is useful only if the model, threshold, and
release rules are fixed before its labels are examined.

**Estimated time:** 50–65 minutes.
**Prerequisite:** lesson 06; you understand the selected candidate, threshold,
average precision, precision, recall, and confusion counts.


## Preflight

Check the kernel and locate or visibly recreate lesson 06 evidence.


In [ ]:
import importlib.util
import sys

required = ("mlflow", "pandas", "sklearn", "aai_local_classification")
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(
        "This notebook is using the wrong Python kernel. Close this Jupyter "
        "server, run `make notebook` from examples/local-classification, or "
        "select the 'AAI Local Classification' kernel. Missing: " + ", ".join(missing)
    )

import pandas as pd

from aai_local_classification.learning import study_root
from aai_local_classification.settings import load_settings
from aai_local_classification.tracking import local_paths

settings = load_settings()
root = study_root()
paths = local_paths(root)
print(f"✓ Python {sys.version_info.major}.{sys.version_info.minor}: {sys.executable}")
print("✓ Course imports are available")
print(f"✓ Learner state: {root}")


In [ ]:
from aai_local_classification.contracts import SplitName
from aai_local_classification.data import load_split
from aai_local_classification.evaluation import recall_slices
from aai_local_classification.learning import state_exists
from aai_local_classification.modeling import feature_frame
from aai_local_classification.workflow import (
    get_or_run_candidate_selection,
    run_frozen_test_gate,
)

selection_was_missing = not state_exists("selection.json")
selection = get_or_run_candidate_selection(settings, root)
chosen = next(
    item for item in selection.candidates if item.run_id == selection.selected_run_id
)
print(
    "Recreated lesson 06 selection"
    if selection_was_missing
    else "✓ Reused lesson 06 selection"
)


### What you should see

Normally, `✓ Reused lesson 06 selection`. If this notebook was opened directly,
it says that it recreated the prerequisite—nothing is silently hidden.

### Words introduced

| Word | Plain meaning | Here |
|---|---|---|
| frozen test | Final rows not used to choose model or threshold | Oct–Dec 2024 |
| release gate | Prewritten pass/fail rules | six checks below |
| adopt / reject | Release may proceed / must not proceed | gate decision |


## Freeze the choice before opening the final exam

**Before you run this:** verify that the selected model, threshold, and all gate
limits come from persisted configuration/evidence—not from test results.


In [ ]:
fixed_choice = pd.Series(
    {
        "candidate": selection.selected_candidate,
        "model_uri": selection.selected_model_uri,
        "threshold_selected_on_validation": chosen.threshold_selection.threshold,
        "minimum_test_average_precision": settings.promotion_gate.minimum_test_average_precision,
        "minimum_test_average_precision_lift": settings.promotion_gate.minimum_test_average_precision_lift,
        "minimum_test_recall": settings.promotion_gate.minimum_test_recall,
        "maximum_test_brier_score": settings.promotion_gate.maximum_test_brier_score,
        "maximum_test_cost_per_1000": settings.promotion_gate.maximum_test_cost_per_1000,
        "maximum_slice_recall_gap": settings.promotion_gate.maximum_slice_recall_gap,
    }
)
fixed_choice.to_frame("fixed before test")


### How to interpret the output

This is the test protocol: one exact logged model, threshold 0.12, and authored
limits. A model change, threshold change, or gate change after viewing test
results would require a new frozen-test dataset version.


## Run the one-time, idempotent gate

**Before you run this:** “adopt” and “reject” are both valid outcomes. Neither
should crash the notebook. Predict which outcome the deterministic course was
designed to demonstrate.


In [ ]:
decision = run_frozen_test_gate(settings, root, selection)
print(f"Decision: {decision.decision.value.upper()}")
print(decision.rationale)


### What you should see

`Decision: ADOPT` and a sentence saying all predeclared checks passed. That means
the artifact passed this teaching gate; it does not prove business value,
fairness, privacy, or universal production readiness.


## Show actual value, rule, and result—not just `True`

**Brier score** is the average squared distance between predicted probability
and a 0/1 outcome. Lower is better; a confidently wrong probability is penalized
more than a cautious one. It is a simple calibration-sensitive diagnostic.


In [ ]:
test_metrics = decision.metrics
gate = settings.promotion_gate
checks = decision.checks.model_dump()

actual_gate_values = pd.Series(
    {
        "average precision": test_metrics["test_average_precision"],
        "AP lift over prevalence": (
            test_metrics["test_average_precision"] - test_metrics["test_positive_rate"]
        ),
        "recall": test_metrics["test_recall"],
        "Brier score": test_metrics["test_brier_score"],
        "cost per 1,000": test_metrics["test_cost_per_1000"],
        "maximum slice recall gap": test_metrics["test_maximum_slice_recall_gap"],
    }
)


The actual values above cover ranking lift, captured positives, probability
quality, operational cost, and concentration by slice. For a no-skill ranking,
AP is near the positive-class prevalence; **AP lift over prevalence** subtracts
that reference rate from observed AP. Next pair each value with the rule that
was fixed before test access.


In [ ]:
required_gate_values = pd.Series(
    {
        "average precision": gate.minimum_test_average_precision,
        "AP lift over prevalence": gate.minimum_test_average_precision_lift,
        "recall": gate.minimum_test_recall,
        "Brier score": gate.maximum_test_brier_score,
        "cost per 1,000": gate.maximum_test_cost_per_1000,
        "maximum slice recall gap": gate.maximum_slice_recall_gap,
    }
)
gate_rules = pd.Series(
    [">=", ">=", ">=", "<=", "<=", "<="],
    index=actual_gate_values.index,
)


The first three metrics have minimums (`>=`); the last three have maximums
(`<=`). Finally, attach the gate's recorded booleans without recalculating or
changing the evidence.


In [ ]:
passed_gate_values = pd.Series(
    list(checks.values()),
    index=actual_gate_values.index,
)
gate_table = pd.DataFrame(
    {
        "actual": actual_gate_values,
        "rule": gate_rules,
        "required": required_gate_values,
        "passed": passed_gate_values,
    }
)
gate_table


### What you should see

Approximately AP 0.471, recall 0.757, Brier 0.135, cost 588.9 per 1,000,
and maximum slice recall gap 0.327. Every row passes its authored comparison.

The margin matters: cost passes 600 but is not far below it. A boolean alone
would hide that operational context.


## Inspect the final confusion counts


In [ ]:
test_confusion = pd.DataFrame(
    [
        [test_metrics["test_true_negatives"], test_metrics["test_false_positives"]],
        [test_metrics["test_false_negatives"], test_metrics["test_true_positives"]],
    ],
    index=["actual 0", "actual 1"],
    columns=["predicted 0", "predicted 1"],
)
test_confusion


### How to interpret the output

The selected threshold catches most positive rows but also flags many negative
rows. The counts make the release trade-off concrete and should be reviewed
with action capacity, not only model developers.


## Compare validation and test without tuning again

Validation chose the model and threshold; test judges that fixed choice. Small
differences are expected on different rows.


In [ ]:
validation_metrics = chosen.threshold_selection.validation_metrics
validation_vs_test = pd.DataFrame(
    {
        "validation": {
            "average_precision": validation_metrics.average_precision,
            "precision": validation_metrics.precision,
            "recall": validation_metrics.recall,
            "brier_score": validation_metrics.brier_score,
            "cost_per_1000": validation_metrics.cost_per_1000,
        },
        "test": {
            "average_precision": test_metrics["test_average_precision"],
            "precision": test_metrics["test_precision"],
            "recall": test_metrics["test_recall"],
            "brier_score": test_metrics["test_brier_score"],
            "cost_per_1000": test_metrics["test_cost_per_1000"],
        },
    }
)
validation_vs_test


The test result is somewhat different but remains inside every fixed gate. We
do not now change the threshold to make the test table prettier.

An **operational slice** groups rows by a field such as plan or signup channel
to reveal concentrated errors. This diagnostic is not a fairness certification.


In [ ]:
import mlflow.sklearn

test = load_split(settings, SplitName.TEST, paths.data_root)
test_model = mlflow.sklearn.load_model(selection.selected_model_uri)
X_test = feature_frame(test, settings)
positive_index = list(test_model.classes_).index(1)
test_probability = test_model.predict_proba(X_test)[:, positive_index]
slice_table = recall_slices(
    test,
    test.churned_30d,
    test_probability,
    decision.threshold,
)
slice_table


### What you should see

Recall for sufficiently large plan/channel groups, including row and positive
counts. A gap can flag investigation, but synthetic categories, limited sample
sizes, and omitted human-impact analysis prevent any fairness conclusion.

### Misconception check

Test is not “validation round two.” If a test result causes a model or policy
change, that new choice has learned from the test and needs new final evidence.


## Prove rerunning does not consume the test twice

The helper is idempotent: compatible evidence is returned rather than creating
a second result run.


In [ ]:
decision_again = run_frozen_test_gate(settings, root, selection)
pd.Series(
    {
        "first_test_run_id": decision.test_run_id,
        "second_test_run_id": decision_again.test_run_id,
        "same_evidence": decision.test_run_id == decision_again.test_run_id,
    }
).to_frame("value")


### Guided exercise

Without reevaluating test rows, apply a hypothetical stricter cost limit of 500
to the already-recorded cost. What release outcome follows?


In [ ]:
exercise_cost_limit = 500.0
exercise_recorded_cost = test_metrics["test_cost_per_1000"]
exercise_passed = exercise_recorded_cost <= exercise_cost_limit
exercise_decision = "adopt" if exercise_passed else "reject"
pd.Series(
    {
        "recorded_cost": exercise_recorded_cost,
        "hypothetical_limit": exercise_cost_limit,
        "hypothetical_decision": exercise_decision,
    }
)


**Self-check:** 588.9 is greater than 500, so the hypothetical outcome is a
valid `reject`. This thought experiment does not rewrite the real gate, create
new test evidence, or support a new release claim. Adopting a stricter gate for
a future release would require new final evidence for the changed policy.

<details><summary>Solution explanation</summary>

A release gate can legitimately reject a statistically promising model because
the selected action policy violates an operational limit.
</details>


In [ ]:
# Reference solution — run after your attempt
assert exercise_decision == "reject"
assert not exercise_passed
print("✓ The stricter hypothetical gate rejects without crashing")


## MLOps bridge

The result run binds the exact logged model, dataset fingerprint, selection
policy, gate policy, threshold, metrics, slices, and decision. In a Databricks
workflow, this gate belongs in a job or CI task—not an informal UI click.

## Recap

- The selected model, threshold, and gate were fixed before test access.
- Actual values, rules, margins, confusion counts, and slices explain the gate.
- Both adopt and reject are valid outcomes; compatible reruns reuse evidence.

**Evidence created:** one MLflow frozen-test result and `decision.json`. The
frozen test is now consumed for this exact model and policy.

**Ready for 08?** You can explain why the passing test is release evidence, not
permission to tune again.
